# 💰 W5: 공사비용 분석 막대그래프
**hexa-5 — 건설헙 AX 마스터클래스 — Week 5**

---
**학습 목표**
1. 건설 공사비용 데이터를 카테고리별로 분석하고 시각화한다
2. 막대그래프를 통해 비용 구조와 예산 대비 실적을 비교 분석한다
3. 비용 최적화 방안과 재무 건전성을 평가한다

> ⏱️ 예상 소요시간: 60분 | 🎯 결과물: 비용분석 대시보드 + 차트 (이미지 다운로드)

## Step 0: 환경 설정

In [ ]:
import subprocess
subprocess.run(['apt-get','-qq','-y','install','fonts-nanum'],capture_output=True)
!pip install -q google-generativeai pandas matplotlib seaborn plotly
import matplotlib.pyplot as plt, matplotlib.font_manager as fm
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

_n=[f for f in fm.findSystemFonts() if 'Nanum' in f]
if _n: fm.fontManager.addfont(_n[0]); plt.rcParams['font.family']=fm.FontProperties(fname=_n[0]).get_name()
plt.rcParams['axes.unicode_minus']=False
try:
    from google.colab import userdata; API_KEY=userdata.get('GEMINI_API_KEY')
except: API_KEY=input('GEMINI_API_KEY: ')
import google.generativeai as genai; genai.configure(api_key=API_KEY)
model=genai.GenerativeModel('gemini-2.0-flash'); print('✅ 준비 완료')

## Step 1: 공사비용 샘플 데이터 생성

In [ ]:
def generate_construction_cost_data():
    """건설 공사비용 샘플 데이터 생성"""
    
    # 프로젝트 목록
    projects = [
        {'project_id': 'P001', 'name': '송파 오피스빌딩', 'location': '서울', 'type': '상업용', 'total_budget': 2500000000},
        {'project_id': 'P002', 'name': '강남 아파트 리모델링', 'location': '서울', 'type': '주거용', 'total_budget': 1800000000},
        {'project_id': 'P003', 'name': '판교 테크밸리 공장', 'location': '경기', 'type': '산업용', 'total_budget': 3200000000}
    ]
    
    # 비용 카테고리 정의
    cost_categories = [
        {'category': '직접재료비', 'subcategories': ['시멘트', '철근', '골재', '내장재', '외장재', '기계설비']},
        {'category': '직접노무비', 'subcategories': ['기능직', '기술직', '관리직', '특수기술']},
        {'category': '경비', 'subcategories': ['장비임차료', '동력비', '수도광열비', '운반비']},
        {'category': '제경비', 'subcategories': ['감독관수당', '일반관리비', '보험료', '세금공과']},
        {'category': '기타비용', 'subcategories': ['설계비', '인허가비', '조경비', '기타잡비']}
    ]
    
    cost_data = []
    
    for project in projects:
        for cat in cost_categories:
            for subcat in cat['subcategories']:
                # 예산 배분 (카테고리별 비율 설정)
                if cat['category'] == '직접재료비':
                    budget_ratio = np.random.uniform(0.15, 0.25)
                elif cat['category'] == '직접노무비':
                    budget_ratio = np.random.uniform(0.12, 0.20)
                elif cat['category'] == '경비':
                    budget_ratio = np.random.uniform(0.08, 0.15)
                elif cat['category'] == '제경비':
                    budget_ratio = np.random.uniform(0.05, 0.12)
                else:  # 기타비용
                    budget_ratio = np.random.uniform(0.03, 0.08)
                
                budget_amount = project['total_budget'] * budget_ratio
                
                # 실제 비용 (예산 대비 ±20% 범위에서 변동)
                actual_variance = np.random.uniform(-0.15, 0.25)
                actual_amount = budget_amount * (1 + actual_variance)
                
                cost_data.append({
                    'project_id': project['project_id'],
                    'project_name': project['name'],
                    'location': project['location'],
                    'project_type': project['type'],
                    'total_budget': project['total_budget'],
                    'category': cat['category'],
                    'subcategory': subcat,
                    'budget_amount': budget_amount,
                    'actual_amount': actual_amount,
                    'variance_amount': actual_amount - budget_amount,
                    'variance_rate': (actual_amount - budget_amount) / budget_amount * 100,
                    'variance_type': '초과' if actual_amount > budget_amount else '절감',
                    'record_date': datetime.now() - timedelta(days=np.random.randint(0, 30))
                })
    
    return pd.DataFrame(cost_data)

# 비용 데이터 생성
df_cost = generate_construction_cost_data()
print(f'💰 공사비용 데이터 생성 완료 (총 {len(df_cost)}건)')
display(df_cost.head())

print('\n📊 데이터 기본 통계:')
print(f'• 프로젝트 수: {df_cost["project_id"].nunique()}개')
print(f'• 비용 카테고리: {df_cost["category"].nunique()}개')
print(f'• 총 예산: {df_cost["budget_amount"].sum():,}원 ({df_cost["budget_amount"].sum()/100000000:.1f}억원)')
print(f'• 총 실제비용: {df_cost["actual_amount"].sum():,}원 ({df_cost["actual_amount"].sum()/100000000:.1f}억원)')
print(f'• 전체 차액: {df_cost["variance_amount"].sum():,}원')

## Step 2: 비용 분석 함수

In [ ]:
def analyze_costs(df):
    """공사비용 종합 분석"""
    
    # 1. 프로젝트별 분석
    project_analysis = df.groupby(['project_id', 'project_name', 'location', 'project_type', 'total_budget']).agg({
        'budget_amount': 'sum',
        'actual_amount': 'sum',
        'variance_amount': 'sum'
    }).reset_index()
    
    project_analysis['variance_rate'] = project_analysis['variance_amount'] / project_analysis['budget_amount'] * 100
    project_analysis['efficiency'] = 100 - abs(project_analysis['variance_rate'])  # 예산 준수도
    project_analysis['performance'] = project_analysis.apply(
        lambda x: '우수' if x['variance_rate'] >= 0 and x['variance_rate'] <= 5 else
                  '양호' if -5 <= x['variance_rate'] < 0 or 5 < x['variance_rate'] <= 10 else
                  '보통' if -10 <= x['variance_rate'] < -5 or 10 < x['variance_rate'] <= 15 else '개선필요', axis=1
    )
    
    # 2. 카테고리별 분석
    category_analysis = df.groupby('category').agg({
        'budget_amount': 'sum',
        'actual_amount': 'sum',
        'variance_amount': 'sum',
        'subcategory': 'count'
    }).rename(columns={'subcategory': 'item_count'})
    
    category_analysis['variance_rate'] = category_analysis['variance_amount'] / category_analysis['budget_amount'] * 100
    category_analysis['budget_ratio'] = category_analysis['budget_amount'] / category_analysis['budget_amount'].sum() * 100
    
    # 3. 상위 초과/절감 항목
    over_budget = df[df['variance_rate'] > 0].sort_values('variance_rate', ascending=False).head(10)
    under_budget = df[df['variance_rate'] < 0].sort_values('variance_rate').head(10)
    
    # 4. 전체 통계
    total_stats = {
        'total_projects': df['project_id'].nunique(),
        'total_budget': df['budget_amount'].sum(),
        'total_actual': df['actual_amount'].sum(),
        'total_variance': df['variance_amount'].sum(),
        'overall_variance_rate': (df['variance_amount'].sum() / df['budget_amount'].sum()) * 100
    }
    
    return project_analysis, category_analysis, over_budget, under_budget, total_stats

# 비용 분석 실행
project_analysis, category_analysis, over_budget, under_budget, total_stats = analyze_costs(df_cost)

print('📊 공사비용 분석 결과:')
print(f'• 전체 예산 대비 실제비용: {(total_stats["total_actual"]/total_stats["total_budget"]*100-100):.1f}%')
print(f'• 총 차액: {total_stats["total_variance"]:,.0f}원 ({total_stats["overall_variance_rate"]:.1f}%)')
print('\n🏢 프로젝트별 성과:')
for _, row in project_analysis.iterrows():
    print(f'• {row["project_name"]}: {row["variance_rate"]:.1f}% ({row["performance"]})')

## Step 3: 막대그래프 시각화 (카테고리별)

In [ ]:
def create_category_bar_charts(category_analysis, df):
    """카테고리별 비용 막대그래프 생성"""
    
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    
    # 1. 예산 대비 실제비용 비교
    categories = category_analysis.index
    x_pos = np.arange(len(categories))
    width = 0.35
    
    bars1 = axes[0,0].bar(x_pos - width/2, category_analysis['budget_amount']/100000000, 
                         width, label='예산', color='skyblue', alpha=0.8)
    bars2 = axes[0,0].bar(x_pos + width/2, category_analysis['actual_amount']/100000000, 
                         width, label='실제', color='lightcoral', alpha=0.8)
    
    axes[0,0].set_xlabel('비용 카테고리', fontsize=12)
    axes[0,0].set_ylabel('금액 (억원)', fontsize=12)
    axes[0,0].set_title('카테고리별 예산 vs 실제비용', fontsize=14, fontweight='bold')
    axes[0,0].set_xticks(x_pos)
    axes[0,0].set_xticklabels(categories, rotation=45)
    axes[0,0].legend()
    axes[0,0].grid(True, alpha=0.3)
    
    # 막대에 수치 표시
    for bar in bars1:
        height = bar.get_height()
        axes[0,0].text(bar.get_x() + bar.get_width()/2., height,
                     f'{height:.1f}', ha='center', va='bottom', fontsize=10)
    for bar in bars2:
        height = bar.get_height()
        axes[0,0].text(bar.get_x() + bar.get_width()/2., height,
                     f'{height:.1f}', ha='center', va='bottom', fontsize=10)
    
    # 2. 차액 분석
    colors = ['red' if x > 0 else 'green' for x in category_analysis['variance_amount']/100000000]
    bars3 = axes[0,1].bar(categories, category_analysis['variance_amount']/100000000, color=colors, alpha=0.7)
    axes[0,1].set_xlabel('비용 카테고리', fontsize=12)
    axes[0,1].set_ylabel('차액 (억원)', fontsize=12)
    axes[0,1].set_title('카테고리별 예산 대비 차액', fontsize=14, fontweight='bold')
    axes[0,1].tick_params(axis='x', rotation=45)
    axes[0,1].axhline(y=0, color='black', linestyle='-', alpha=0.5)
    axes[0,1].grid(True, alpha=0.3)
    
    # 막대에 수치 표시
    for bar in bars3:
        height = bar.get_height()
        axes[0,1].text(bar.get_x() + bar.get_width()/2., height,
                     f'{height:.1f}', ha='center', va='bottom' if height > 0 else 'top', fontsize=10)
    
    # 3. 비용 구성 비율
    wedges, texts, autotexts = axes[1,0].pie(category_analysis['budget_ratio'], labels=categories, 
                                        autopct='%1.1f%%', startangle=90, textprops={'fontsize': 10})
    axes[1,0].set_title('예산 구성 비율', fontsize=14, fontweight='bold')
    
    # 4. 변동률 분석
    colors2 = ['red' if x > 5 else 'orange' if 0 <= x <= 5 else 'green' if x < 0 else 'blue' 
               for x in category_analysis['variance_rate']]
    bars4 = axes[1,1].bar(categories, category_analysis['variance_rate'], color=colors2, alpha=0.7)
    axes[1,1].set_xlabel('비용 카테고리', fontsize=12)
    axes[1,1].set_ylabel('변동률 (%)', fontsize=12)
    axes[1,1].set_title('카테고리별 예산 변동률', fontsize=14, fontweight='bold')
    axes[1,1].tick_params(axis='x', rotation=45)
    axes[1,1].axhline(y=0, color='black', linestyle='-', alpha=0.5)
    axes[1,1].axhline(y=5, color='red', linestyle='--', alpha=0.5, label='관리상한')
    axes[1,1].axhline(y=-5, color='green', linestyle='--', alpha=0.5, label='관리하한')
    axes[1,1].legend()
    axes[1,1].grid(True, alpha=0.3)
    
    # 막대에 수치 표시
    for bar in bars4:
        height = bar.get_height()
        axes[1,1].text(bar.get_x() + bar.get_width()/2., height,
                     f'{height:.1f}%', ha='center', va='bottom' if height > 0 else 'top', fontsize=10)
    
    plt.tight_layout()
    plt.show()

# 막대그래프 생성
create_category_bar_charts(category_analysis, df_cost)

## Step 4: 프로젝트별 비용 분석 차트

In [ ]:
def create_project_cost_charts(project_analysis):
    """프로젝트별 비용 분석 차트 생성"""
    
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    
    # 1. 프로젝트별 예산 대비 실제비용
    project_names = project_analysis['project_name']
    x_pos = np.arange(len(project_names))
    width = 0.35
    
    bars1 = axes[0,0].bar(x_pos - width/2, project_analysis['budget_amount']/100000000, 
                         width, label='예산', color='lightblue', alpha=0.8)
    bars2 = axes[0,0].bar(x_pos + width/2, project_analysis['actual_amount']/100000000, 
                         width, label='실제', color='salmon', alpha=0.8)
    
    axes[0,0].set_xlabel('프로젝트', fontsize=12)
    axes[0,0].set_ylabel('금액 (억원)', fontsize=12)
    axes[0,0].set_title('프로젝트별 예산 vs 실제비용', fontsize=14, fontweight='bold')
    axes[0,0].set_xticks(x_pos)
    axes[0,0].set_xticklabels(project_names, rotation=15)
    axes[0,0].legend()
    axes[0,0].grid(True, alpha=0.3)
    
    # 막대에 수치 표시
    for bar in bars1:
        height = bar.get_height()
        axes[0,0].text(bar.get_x() + bar.get_width()/2., height,
                     f'{height:.0f}', ha='center', va='bottom', fontsize=10)
    for bar in bars2:
        height = bar.get_height()
        axes[0,0].text(bar.get_x() + bar.get_width()/2., height,
                     f'{height:.0f}', ha='center', va='bottom', fontsize=10)
    
    # 2. 프로젝트별 변동률
    colors = ['red' if x > 5 else 'orange' if 0 <= x <= 5 else 'green' if x < 0 else 'blue' 
               for x in project_analysis['variance_rate']]
    bars3 = axes[0,1].bar(project_names, project_analysis['variance_rate'], color=colors, alpha=0.7)
    axes[0,1].set_xlabel('프로젝트', fontsize=12)
    axes[0,1].set_ylabel('변동률 (%)', fontsize=12)
    axes[0,1].set_title('프로젝트별 예산 변동률', fontsize=14, fontweight='bold')
    axes[0,1].tick_params(axis='x', rotation=15)
    axes[0,1].axhline(y=0, color='black', linestyle='-', alpha=0.5)
    axes[0,1].axhline(y=5, color='red', linestyle='--', alpha=0.5, label='관리상한')
    axes[0,1].axhline(y=-5, color='green', linestyle='--', alpha=0.5, label='관리하한')
    axes[0,1].legend()
    axes[0,1].grid(True, alpha=0.3)
    
    # 막대에 수치 표시
    for bar in bars3:
        height = bar.get_height()
        axes[0,1].text(bar.get_x() + bar.get_width()/2., height,
                     f'{height:.1f}%', ha='center', va='bottom' if height > 0 else 'top', fontsize=10)
    
    # 3. 효율성 점수
    colors2 = ['gold' if x >= 95 else 'lightgreen' if x >= 90 else 'orange' if x >= 85 else 'lightcoral' 
                for x in project_analysis['efficiency']]
    bars4 = axes[1,0].bar(project_names, project_analysis['efficiency'], color=colors2, alpha=0.7)
    axes[1,0].set_xlabel('프로젝트', fontsize=12)
    axes[1,0].set_ylabel('효율성 점수', fontsize=12)
    axes[1,0].set_title('프로젝트별 예산 준수 효율성', fontsize=14, fontweight='bold')
    axes[1,0].tick_params(axis='x', rotation=15)
    axes[1,0].set_ylim(0, 100)
    axes[1,0].grid(True, alpha=0.3)
    
    # 막대에 수치 표시
    for bar in bars4:
        height = bar.get_height()
        axes[1,0].text(bar.get_x() + bar.get_width()/2., height + 1,
                     f'{height:.1f}', ha='center', va='bottom', fontsize=10)
    
    # 4. 성과 등급 분포
    performance_counts = project_analysis['performance'].value_counts()
    colors3 = ['green', 'lightblue', 'orange', 'red']
    wedges, texts, autotexts = axes[1,1].pie(performance_counts.values, labels=performance_counts.index, 
                                        autopct='%1.1f%%', colors=colors3, startangle=90, textprops={'fontsize': 12})
    axes[1,1].set_title('프로젝트 성과 등급 분포', fontsize=14, fontweight='bold')
    
    plt.tight_layout()
    plt.show()

# 프로젝트별 차트 생성
create_project_cost_charts(project_analysis)

## Step 5: 상세 항목별 분석

In [ ]:
def analyze_detailed_items(df):
    """상세 비용 항목별 분석"""
    
    # 상위 초과 항목
    top_over = df[df['variance_rate'] > 0].nlargest(15, 'variance_rate')
    
    # 상위 절감 항목
    top_under = df[df['variance_rate'] < 0].nsmallest(15, 'variance_rate')
    
    # 카테고리별 상세 분석
    category_details = {}
    for category in df['category'].unique():
        cat_data = df[df['category'] == category]
        category_details[category] = {
            'total_variance': cat_data['variance_amount'].sum(),
            'variance_rate': (cat_data['variance_amount'].sum() / cat_data['budget_amount'].sum()) * 100,
            'max_over_item': cat_data[cat_data['variance_rate'] > 0].nlargest(1, 'variance_rate'),
            'max_under_item': cat_data[cat_data['variance_rate'] < 0].nsmallest(1, 'variance_rate')
        }
    
    return top_over, top_under, category_details

# 상세 항목 분석
top_over, top_under, category_details = analyze_detailed_items(df_cost)

print('🔝 상위 예산 초과 항목 (TOP 10):')
if not top_over.empty:
    display(top_over[['project_name', 'category', 'subcategory', 'variance_rate', 'variance_amount']].head(10)
               .style.format({'variance_rate': '{:.1f}%', 'variance_amount': '{:,.0f}원'}))
else:
    print('예산 초과 항목이 없습니다.')

print('\n🔽 상위 예산 절감 항목 (TOP 10):')
if not top_under.empty:
    display(top_under[['project_name', 'category', 'subcategory', 'variance_rate', 'variance_amount']].head(10)
               .style.format({'variance_rate': '{:.1f}%', 'variance_amount': '{:,.0f}원'}))
else:
    print('예산 절감 항목이 없습니다.')

print('\n📂 카테고리별 특이사항:')
for category, details in category_details.items():
    print(f'\n• {category}:')
    print(f'  - 총 차액: {details["total_variance"]:,.0f}원 ({details["variance_rate"]:.1f}%)')
    if not details['max_over_item'].empty:
        item = details['max_over_item'].iloc[0]
        print(f'  - 최대 초과: {item["subcategory"]} ({item["variance_rate"]:.1f}%)')
    if not details['max_under_item'].empty:
        item = details['max_under_item'].iloc[0]
        print(f'  - 최대 절감: {item["subcategory"]} ({item["variance_rate"]:.1f}%)')

## Step 6: AI 기반 비용 최적화 제안

In [ ]:
def generate_cost_optimization_suggestions(df, project_analysis, category_analysis):
    """Gemini AI로 비용 최적화 제안 생성"""
    
    # 주요 분석 데이터
    problematic_categories = category_analysis[category_analysis['variance_rate'] > 5].sort_values('variance_rate', ascending=False)
    efficient_categories = category_analysis[category_analysis['variance_rate'] < -5].sort_values('variance_rate')
    problematic_projects = project_analysis[project_analysis['variance_rate'] > 10]
    
    prompt = f"""
당신은 건설 비용 관리 전문가입니다. 다음 공사비용 분석 결과를 바탕으로 구체적인 비용 최적화 전략을 제안해주세요.

## 전체 현황
- 프로젝트 수: {len(project_analysis)}개
- 전체 예산: {total_stats['total_budget']:,.0f}원 ({total_stats['total_budget']/100000000:.1f}억원)
- 전체 실제비용: {total_stats['total_actual']:,.0f}원 ({total_stats['total_actual']/100000000:.1f}억원)
- 전체 차액: {total_stats['total_variance']:,.0f}원 ({total_stats['overall_variance_rate']:.1f}%)

## 문제 카테고리 (예산 초과 5% 이상)
{problematic_categories[['budget_amount', 'actual_amount', 'variance_rate']].to_string(float_format='%.1f') if not problematic_categories.empty else '문제 카테고리 없음'}

## 효율 카테고리 (예산 절감 5% 이상)
{efficient_categories[['budget_amount', 'actual_amount', 'variance_rate']].to_string(float_format='%.1f') if not efficient_categories.empty else '효율 카테고리 없음'}

## 문제 프로젝트 (예산 초과 10% 이상)
{problematic_projects[['project_name', 'variance_rate', 'variance_amount']].to_string(float_format='%.1f') if not problematic_projects.empty else '문제 프로젝트 없음'}

## 분석 요구사항
1. **원인 분석**: 비용 초과의 근본 원인 (재료가격, 인건비, 장비임대 등)
2. **단기 대책**: 즉시 적용 가능한 비용 절감 방안
3. **장기 전략**: 재발 방지를 위한 시스템적 개선 방안
4. **구매 전략**: 재료비 절감을 위한 구매 최적화 전략
5. **노무비 관리**: 인건비 효율화 방안
6. **장비 관리**: 장비 임대료 최적화 방안
7. **예산 관리**: 더 정확한 예산 수립 방법론

## 출력 형식
- 마크다운 형식
- 구체적이고 실용적인 제안
- 실행 가능성 고려한 우선순위 제시
- ROI 기반의 효과 예상

건설 현장 실무 경험을 바탕으로 실질적인 비용 최적화 전략을 제시해주세요.
"""
    
    try:
        response = model.generate_content(prompt)
        return response.text
    except Exception as e:
        print(f"AI 분석 중 오류 발생: {e}")
        return "AI 분석을 통한 비용 최적화 제안을 생성할 수 없습니다."

# AI 비용 최적화 제안 생성
print('💰 AI 기반 비용 최적화 전략:')
print('=' * 60)
optimization_suggestions = generate_cost_optimization_suggestions(df_cost, project_analysis, category_analysis)
print(optimization_suggestions)

## Step 7: 비용분석 보고서 생성 및 저장

In [ ]:
def generate_cost_analysis_report(project_analysis, category_analysis, total_stats, optimization_suggestions):
    """공사비용 분석 종합 보고서 생성"""
    
    report_date = datetime.now().strftime('%Y년 %m월 %d일')
    
    report = f"""
# 공사비용 분석 종합 보고서

> 보고일: {report_date}
> 분석 대상: {len(project_analysis)}개 프로젝트

## 1. 개요

본 보고서는 건설헙 사의 비용 관리 실태를 분석하고, 예산 준수도와 효율성을 평가한 결과입니다.
비용 최적화 방안과 재무 건전성 향상 전략을 제시합니다.

## 2. 종합 평가

| 구분 | 금액 | 비율 | 평가 |
|------|------|------|------|
| 총 예산 | {total_stats['total_budget']:,.0f}원 | 100% | - |
| 실제비용 | {total_stats['total_actual']:,.0f}원 | {total_stats['total_actual']/total_stats['total_budget']*100:.1f}% | {'절감' if total_stats['total_variance'] < 0 else '초과'} |
| 차액 | {abs(total_stats['total_variance']):,.0f}원 | {abs(total_stats['overall_variance_rate']):.1f}% | {'양호' if abs(total_stats['overall_variance_rate']) <= 5 else '보통' if abs(total_stats['overall_variance_rate']) <= 10 else '개선필요'} |

## 3. 프로젝트별 성과

{project_analysis[['project_name', 'variance_rate', 'performance']].to_string(index=False, float_format='%.1f')}

### 프로젝트별 특징
"""
    
    # 프로젝트별 특징 추가
    for _, project in project_analysis.iterrows():
        perf_level = {'우수': '예산 관리가 매우 우수하여 타 프로젝트의 모범 사례로 적합함',
                      '양호': '예산 관리가 양호하나 일부 개선이 필요함',
                      '보통': '예산 관리에 시급한 개선이 필요함',
                      '개선필요': '비용 관리 시스템 전면 개선이 시급함'}[project['performance']]
        
        report += f"""
- **{project['project_name']}**: 변동률 {project['variance_rate']:.1f}% ({project['performance']}) - {perf_level}
"""
    
    report += f"""

## 4. 카테고리별 분석

{category_analysis[['budget_amount', 'actual_amount', 'variance_rate']].to_string(float_format='%.1f')}

## 5. 주요 비용 이슈

### 예산 초과 상위 항목
"""
    
    if not top_over.empty:
        report += top_over.head(5)[['project_name', 'category', 'subcategory', 'variance_rate']].to_string(index=False, float_format='%.1f')
    else:
        report += "\n예산 초과 항목이 없어 비용 관리가 양호합니다."
    
    report += f"""

### 예산 절감 상위 항목
"""
    
    if not top_under.empty:
        report += top_under.head(5)[['project_name', 'category', 'subcategory', 'variance_rate']].to_string(index=False, float_format='%.1f')
    else:
        report += "\n예산 절감 항목이 없습니다."
    
    report += f"""

## 6. AI 기반 비용 최적화 전략

{optimization_suggestions}

## 7. 개선 과제 및 실행 계획

### 7.1 단기 실행 과제 (1개월 이내)
1. 예산 초과 10% 이상 프로젝트 즉각적인 원인 분석
2. 직접재료비 구매 프로세스 재검토 및 개선
3. 장비 임대료 계약 재협상
4. 일일 비용 추적 시스템 강화

### 7.2 중기 실행 과제 (3개월 이내)
1. 공종별 표준 단가 재설정
2. 공급망 관리 시스템 구축
3. 기술 인력 효율화 프로그램 도입
4. 예산 관리 교육 프로그램 실시

### 7.3 장기 실행 과제 (6개월 이내)
1. 스마트 비용 관리 플랫폼 구축
2. 데이터 기반 예산 예측 시스템 도입
3. 지속가능한 파트너십 구축
4. 비용 관리 성과평가제도 정착

## 8. 기대 효과

- **단기**: 비용 절감율 5~10% 개선
- **중기**: 예산 준수율 95% 이상 달성
- **장기**: 지속가능한 비용 관리 시스템 구축

## 9. 결론

전반적인 비용 관리 수준은 {'우수하나' if abs(total_stats['overall_variance_rate']) <= 5 else '양호하나' if abs(total_stats['overall_variance_rate']) <= 10 else '개선이 필요합니다.'}
특히 {len(project_analysis[project_analysis['variance_rate'] > 10])}개 프로젝트에서 시급한 개선이 필요합니다.

본 보고서의 제언사항을 적극 반영하여 재무 건전성을 강화하고 경쟁력을 제고해야 합니다.

---
*본 보고서는 AI 자동 분석 시스템을 통해 생성되었습니다.*
"""
    
    return report

# 비용분석 보고서 생성
cost_report = generate_cost_analysis_report(project_analysis, category_analysis, total_stats, optimization_suggestions)

# 보고서 저장
with open('공사비용_분석_보고서.md', 'w', encoding='utf-8') as f:
    f.write(cost_report)

print('📋 공사비용 분석 보고서 생성 완료!')
print('📁 파일명: 공사비용_분석_보고서.md')
print('\n📄 보고서 상단 500자 미리보기:')
print(cost_report[:500] + '...')

## Step 8: 결과 저장 및 다운로드

In [ ]:
import os
import zipfile

# 저장할 디렉토리 생성
!mkdir -p cost_analysis_results

# 1. 상세 비용 데이터 저장
detailed_cost = df_cost.copy()
detailed_cost['record_date'] = detailed_cost['record_date'].dt.strftime('%Y-%m-%d')
detailed_cost.to_excel('cost_analysis_results/상세_비용_데이터.xlsx', index=False, encoding='utf-8-sig')

# 2. 프로젝트별 요약 저장
project_summary = project_analysis.copy()
project_summary.to_excel('cost_analysis_results/프로젝트별_요약.xlsx', index=False, encoding='utf-8-sig')

# 3. 카테고리별 분석 저장
category_summary = category_analysis.reset_index()
category_summary.to_excel('cost_analysis_results/카테고리별_분석.xlsx', index=False, encoding='utf-8-sig')

# 4. 문제 항목 목록 저장
if not top_over.empty:
    top_over[['project_name', 'category', 'subcategory', 'variance_rate', 'variance_amount']].to_excel(
        'cost_analysis_results/예산_초과_항목.xlsx', index=False, encoding='utf-8-sig'
    )

if not top_under.empty:
    top_under[['project_name', 'category', 'subcategory', 'variance_rate', 'variance_amount']].to_excel(
        'cost_analysis_results/예산_절감_항목.xlsx', index=False, encoding='utf-8-sig'
    )

# 5. 통계 요약 저장
stats_summary = {
    '총 예산': [total_stats['total_budget']],
    '실제비용': [total_stats['total_actual']],
    '총 차액': [total_stats['total_variance']],
    '변동률': [total_stats['overall_variance_rate']],
    '프로젝트 수': [len(project_analysis)],
    '보고 생성일': [datetime.now().strftime('%Y-%m-%d')]
}

pd.DataFrame(stats_summary).to_excel('cost_analysis_results/통계_요약.xlsx', index=False, encoding='utf-8-sig')

# ZIP 파일 생성
with zipfile.ZipFile('공사비용_분석_결과_모음.zip', 'w') as zipf:
    for foldername, subfolders, filenames in os.walk('cost_analysis_results'):
        for filename in filenames:
            filepath = os.path.join(foldername, filename)
            zipf.write(filepath, filename)
    
    # 보고서 파일도 추가
    zipf.write('공사비용_분석_보고서.md', '공사비용_분석_보고서.md')

# 생성된 파일 목록
print('✅ 모든 비용분석 결과 저장 완료!')
print('\n📁 생성된 파일 목록:')
for filename in os.listdir('cost_analysis_results'):
    print(f'  - {filename}')

# 다운로드
try:
    from google.colab import files
    files.download('공사비용_분석_결과_모음.zip')
    files.download('공사비용_분석_보고서.md')
    print('\n🎉 파일 다운로드 완료!')
except:
    print('\n⚠️ 다운로드 기능을 사용할 수 없습니다.')

# 최종 요약
print('\n🎯 최종 요약:')
print(f'• 총 {len(project_analysis)}개 프로젝트 비용분석 완료')
print(f'• 전체 변동률: {total_stats["overall_variance_rate"]:.1f}%')
print(f'• 효율적 프로젝트: {len(project_analysis[project_analysis["efficiency"] >= 90])}개')
print('• AI 기반 비용 최적화 전략 제안 완료')